# Optical Response of Copper — Drude Model
Visualizes $\sigma(\omega)$, phase angle, dissipated power, $\epsilon(\omega)$, and reflectivity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Constants
e    = 1.6e-19    # C
m    = 9.1e-31    # kg
eps0 = 8.85e-12   # F/m
hbar = 1.055e-34  # J·s
eV   = 1.6e-19    # J per eV

# Copper parameters
n   = 8.5e28   # m^-3
tau = 2.5e-14  # s

# Derived quantities
omega_p = np.sqrt(n * e**2 / (eps0 * m))
sigma0  = n * e**2 * tau / m

hbar_over_tau_eV = (hbar / tau) / eV
hbar_omega_p_eV  = (hbar * omega_p) / eV

print(f"omega_p       = {omega_p:.3e} rad/s")
print(f"hbar/tau      = {hbar_over_tau_eV:.4f} eV")
print(f"hbar*omega_p  = {hbar_omega_p_eV:.2f} eV")

In [ ]:
# Frequency axis (log scale in eV)
hbar_omega_eV = np.logspace(-3, 2, 2000)
omega = hbar_omega_eV * eV / hbar

# Complex conductivity
sigma  = sigma0 / (1 - 1j * omega * tau)
sigma1 = np.real(sigma)
sigma2 = np.imag(sigma)

# Phase angle
phi = np.arctan2(sigma2, sigma1)

# Dissipated power
P_ratio = sigma1 / sigma0

# Complex dielectric function
epsilon = 1 + 1j * sigma / (eps0 * omega)
eps1 = np.real(epsilon)
eps2 = np.imag(epsilon)

# Reflectivity
sqrt_eps = np.sqrt(epsilon.astype(complex))
R = np.abs((sqrt_eps - 1) / (sqrt_eps + 1))**2

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
colors = {'bg': '#0d1117', 'panel': '#161b22', 'grid': '#21262d',
          'c1': '#58a6ff', 'c2': '#ff7b72', 'c3': '#3fb950', 'text': '#e6edf3'}

fig = plt.figure(figsize=(14, 10))
fig.patch.set_facecolor(colors['bg'])
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

axes = [fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
        fig.add_subplot(gs[2, :])]

for ax in axes:
    ax.set_facecolor(colors['panel'])
    ax.tick_params(colors=colors['text'], labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor(colors['grid'])
    ax.grid(True, color=colors['grid'], linewidth=0.5, alpha=0.8)
    ax.set_xscale('log')

def mark_scales(ax, ymin, ymax):
    ax.axvline(hbar_over_tau_eV, color='#f0e68c', lw=1.2, ls='--', alpha=0.8)
    ax.axvline(hbar_omega_p_eV,  color='#da70d6', lw=1.2, ls='--', alpha=0.8)
    ax.text(hbar_over_tau_eV * 1.08, ymax * 0.82, r'$\hbar/\tau$',  color='#f0e68c', fontsize=8)
    ax.text(hbar_omega_p_eV  * 1.08, ymax * 0.82, r'$\hbar\omega_p$', color='#da70d6', fontsize=8)

def set_xlabel(ax):
    ax.set_xlabel(r'$\hbar\omega$ (eV)', color=colors['text'], fontsize=9)

# (i) Complex conductivity
ax = axes[0]
ax.plot(hbar_omega_eV, sigma1/sigma0, color=colors['c1'], lw=1.5, label=r'$\sigma_1/\sigma_0$')
ax.plot(hbar_omega_eV, sigma2/sigma0, color=colors['c2'], lw=1.5, label=r'$\sigma_2/\sigma_0$')
ax.set_ylim(-0.1, 1.1)
ax.set_ylabel(r'$\sigma/\sigma_0$', color=colors['text'], fontsize=9)
ax.set_title('(i) Complex Conductivity', color=colors['text'], fontsize=10)
ax.legend(fontsize=8, facecolor=colors['panel'], labelcolor=colors['text'], edgecolor=colors['grid'])
mark_scales(ax, -0.1, 1.1)
set_xlabel(ax)

# (ii) Phase angle
ax = axes[1]
ax.plot(hbar_omega_eV, np.degrees(phi), color=colors['c3'], lw=1.5)
ax.set_ylabel(r'$\phi(\omega)$ (degrees)', color=colors['text'], fontsize=9)
ax.set_title(r'(ii) Phase Angle $\phi = \arctan(\sigma_2/\sigma_1)$', color=colors['text'], fontsize=10)
mark_scales(ax, np.degrees(phi).min(), np.degrees(phi).max())
set_xlabel(ax)

# (iii) Dissipated power
ax = axes[2]
ax.plot(hbar_omega_eV, P_ratio, color=colors['c1'], lw=1.5)
ax.set_ylim(-0.05, 1.1)
ax.set_ylabel(r'$\sigma_1/\sigma_0$', color=colors['text'], fontsize=9)
ax.set_title('(iii) Dissipated Power $P(\omega)/P(0)$', color=colors['text'], fontsize=10)
mark_scales(ax, -0.05, 1.1)
set_xlabel(ax)

# (iv) Dielectric function
ax = axes[3]
ax.plot(hbar_omega_eV, eps1, color=colors['c1'], lw=1.5, label=r'$\epsilon_1$')
ax.plot(hbar_omega_eV, eps2, color=colors['c2'], lw=1.5, label=r'$\epsilon_2$')
ax.axhline(0, color=colors['text'], lw=0.5, alpha=0.5)
ax.set_ylim(-50, 50)
ax.set_ylabel(r'$\epsilon(\omega)$', color=colors['text'], fontsize=9)
ax.set_title('(iv) Complex Dielectric Function', color=colors['text'], fontsize=10)
ax.legend(fontsize=8, facecolor=colors['panel'], labelcolor=colors['text'], edgecolor=colors['grid'])
mark_scales(ax, -50, 50)
set_xlabel(ax)

# (v) Reflectivity
ax = axes[4]
ax.plot(hbar_omega_eV, R, color=colors['c3'], lw=1.8)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel(r'$R(\omega)$', color=colors['text'], fontsize=9)
ax.set_title(r'(v) Reflectivity $R = \left|\frac{\sqrt{\epsilon}-1}{\sqrt{\epsilon}+1}\right|^2$',
             color=colors['text'], fontsize=10)
mark_scales(ax, -0.05, 1.05)
set_xlabel(ax)

fig.suptitle('Optical Response of Copper — Drude Model', color=colors['text'],
             fontsize=13, fontweight='bold', y=0.98)
plt.show()